In [3]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenckDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits for rank in self.ranks]

    def __len__(self):
        return len(self._cards)
    
    def __getitem__(self, position):
        return self._cards[position]

In [2]:
from random import shuffle

In [3]:
def set_card(deck, position, card):
    deck._cards[position] = card

FrenckDeck.__setitem__ = set_card

In [5]:
deck = FrenckDeck()
deck[:5]

[Card(rank='2', suit='spades'),
 Card(rank='3', suit='spades'),
 Card(rank='4', suit='spades'),
 Card(rank='5', suit='spades'),
 Card(rank='6', suit='spades')]

In [6]:
shuffle(deck)
deck[:5]

[Card(rank='10', suit='diamonds'),
 Card(rank='7', suit='diamonds'),
 Card(rank='7', suit='clubs'),
 Card(rank='8', suit='diamonds'),
 Card(rank='Q', suit='diamonds')]

In [1]:
from collections import abc

In [4]:
class FrenckDeck2(abc.MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits for rank in self.ranks]
    
    def __len__(self):
        return len(self._cards)
    
    def __getitem__(self, index):
        return self._cards[index]
    
    def __setitem__(self, index, value):
        self._cards[index] = value

    def __delitem__(self, index):
        del self._cards[index]
    
    def insert(self, index, value):
        self._cards.insert(index, value)

In [5]:
deck2 = FrenckDeck2()

In [6]:
deck2.pop()

Card(rank='A', suit='hearts')

In [7]:
deck2.append(deck2.pop())

In [4]:
import abc
import random

In [3]:
class Tombola(abc.ABC):
    @abc.abstractmethod
    def load(self, iterable):
        """Add items from an iterable."""

    @abc.abstractmethod
    def pick(self):
        """Remove item at random, returning it.
        
        This method should raise `LookupError` when then instance is empty.
        """
    
    def loaded(self):
        """Return `True` if there's at least 1 item, `False` otherwise."""
        return bool(self.inspect())
    
    def inspect(self):
        """Return a sorted tuple with the items currently inside."""
        items = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        
        self.load(items)
        return tuple(items)

In [5]:
class BingoCage(Tombola):
    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)
    
    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')
        
    def __call__(self):
        self.pick()

In [6]:
class LottoBlower(Tombola):
    def __init__(self, iterable):
        self._balls = list(iterable)
    
    def load(self, iterable):
        self._balls.extend(iterable)
    
    def pick(self):
        try:
            position = random.randrange(len(self._balls))
        except ValueError:
            raise LookupError('pick from empty LottoBlower')
        return self._balls.pop(position)
    
    def loaded(self):
        return bool(self._balls)
    
    def inspect(self):
        return tuple(self._balls)